# SAO RPG — Trilhas com MusicGen-Large (Google Colab)

Mesma lógica de qualidade usada localmente (retry anti-silêncio, encadeamento por continuação de áudio pra não trocar de "banda" no meio, loop sem costura), só que com o **musicgen-large** (3.3B, melhor fidelidade/menos artefato metálico que o medium) numa GPU melhor que a local.

**Como usar:**
1. Menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU** (T4 já serve).
2. Rode as células em ordem (▶ em cada uma, de cima pra baixo).
3. A célula final gera as 6 faixas do projeto e salva no seu Google Drive em `SAO_RPG_musicas_large/`.
4. Baixe a pasta do Drive e substitua os arquivos em `SAO RPG/musicas/` no seu PC.

In [ ]:
!pip install -q transformers accelerate scipy

In [ ]:
import torch
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("ATENCAO: sem GPU. Va em Ambiente de execucao > Alterar tipo de ambiente de execucao > GPU")

## Biblioteca de geração (mesma lógica do projeto local, adaptada pro `musicgen-large`)

In [ ]:
import time
from collections import deque

import numpy as np
from transformers import AutoProcessor, MusicgenForConditionalGeneration

MODEL_ID = "facebook/musicgen-large"
TOKENS_PER_SEC = 51.2
WIN_S = 0.1
# mesmo teto conservador validado localmente com o medium -- mantido aqui
# por seguranca (posicao do decoder pode nao aguentar mais que isso).
MAX_TOTAL_TOKENS = 1843

_model = None
_processor = None


def get_model():
    global _model
    if _model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Carregando {MODEL_ID} ({device.upper()})...")
        _model = MusicgenForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        ).to(device)
    return _model


def get_processor():
    global _processor
    if _processor is None:
        _processor = AutoProcessor.from_pretrained(MODEL_ID)
    return _processor


def _device():
    return next(get_model().parameters()).device


def _sampling_rate():
    return get_model().config.audio_encoder.sampling_rate


def _rms_windows(audio, sr, win_s=WIN_S):
    win = int(sr * win_s)
    n = len(audio) // win
    return np.array([np.sqrt(np.mean(audio[i * win:(i + 1) * win] ** 2)) for i in range(n)]), win


def _sliding_min(rms, k):
    n = len(rms)
    if n < k:
        return np.array([])
    dq = deque()
    mins = np.zeros(n - k + 1)
    for i in range(n):
        while dq and rms[dq[-1]] >= rms[i]:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            mins[i - k + 1] = rms[dq[0]]
    return mins


def _best_window(audio, sr, duration, quality_ratio):
    rms, win = _rms_windows(audio, sr)
    loud_ref = np.percentile(rms, 80) if len(rms) else 0
    k = int(duration / WIN_S)
    mins = _sliding_min(rms, k)
    if len(mins) == 0:
        return None, 0.0
    start_idx = int(np.argmax(mins))
    window_min = mins[start_idx]
    ratio = window_min / loud_ref if loud_ref > 0 else 0
    start_sample = start_idx * win
    end_sample = start_sample + int(duration * sr)
    return audio[start_sample:end_sample], ratio


def _fade_edges(audio, sr, fade_s=0.05):
    fade_n = int(sr * fade_s)
    audio = audio.copy()
    audio[:fade_n] *= np.linspace(0, 1, fade_n)
    audio[-fade_n:] *= np.linspace(1, 0, fade_n)
    return audio


def _run_generate(inputs, tokens, guidance_scale, seed):
    model = get_model()
    device = _device()
    dtype = next(model.parameters()).dtype
    cast = {}
    prompt_tokens = 0
    for k, v in inputs.items():
        v = v.to(device)
        if v.is_floating_point():
            v = v.to(dtype)
        cast[k] = v
        if k == "input_values":
            prompt_tokens = int(v.shape[-1] / _sampling_rate() * TOKENS_PER_SEC)
    tokens = min(tokens, max(1, MAX_TOTAL_TOKENS - prompt_tokens))
    if seed is not None:
        torch.manual_seed(seed)
    out = model.generate(**cast, max_new_tokens=tokens, guidance_scale=guidance_scale)
    audio = out[0, 0].to(torch.float32).cpu().numpy()
    return audio, _sampling_rate()


def generate_clean_segment(prompt, duration, guidance_scale=4.5, max_retries=3, buffer_s=6, seed=None, quality_ratio=0.3):
    processor = get_processor()
    best = None
    best_min_level = -1
    for attempt in range(max_retries):
        tokens = int(round((duration + buffer_s) * TOKENS_PER_SEC))
        t0 = time.time()
        inputs = processor(text=[prompt], padding=True, return_tensors="pt")
        audio, sr = _run_generate(inputs, tokens, guidance_scale, seed + attempt if seed is not None else None)
        gen_t = time.time() - t0
        candidate, ratio = _best_window(audio, sr, duration, quality_ratio)
        if candidate is None:
            print(f"    tentativa {attempt+1}/{max_retries}: gen={gen_t:.1f}s status=curto_demais")
            continue
        status = "ok" if ratio >= quality_ratio else "fraco"
        print(f"    tentativa {attempt+1}/{max_retries}: gen={gen_t:.1f}s status={status} ratio={ratio:.2f}")
        if ratio > best_min_level:
            best_min_level = ratio
            best = (candidate, sr)
        if ratio >= quality_ratio:
            break
    if best is None:
        raise RuntimeError(f"Nao consegui gerar segmento limpo para o prompt: {prompt}")
    trimmed, sr = best
    target_len = int(duration * sr)
    if len(trimmed) < target_len:
        trimmed = np.pad(trimmed, (0, target_len - len(trimmed)))
    return _fade_edges(trimmed, sr), sr


def generate_continuation_segment(prompt, duration, audio_anterior, sr_anterior, guidance_scale=4.5, max_retries=3,
                                   buffer_s=5, seed=None, quality_ratio=0.3, continuation_s=3):
    processor = get_processor()
    trecho = audio_anterior[-int(continuation_s * sr_anterior):]
    best = None
    best_min_level = -1
    for attempt in range(max_retries):
        tokens = int(round((duration + buffer_s) * TOKENS_PER_SEC))
        t0 = time.time()
        inputs = processor(audio=trecho, sampling_rate=sr_anterior, text=[prompt], padding=True, return_tensors="pt")
        audio, sr = _run_generate(inputs, tokens, guidance_scale, seed + attempt if seed is not None else None)
        gen_t = time.time() - t0
        corte = int(continuation_s * sr)
        novo = audio[corte:] if len(audio) > corte else audio
        candidate, ratio = _best_window(novo, sr, duration, quality_ratio)
        if candidate is None:
            print(f"    tentativa {attempt+1}/{max_retries}: gen={gen_t:.1f}s status=curto_demais")
            continue
        status = "ok" if ratio >= quality_ratio else "fraco"
        print(f"    tentativa {attempt+1}/{max_retries}: gen={gen_t:.1f}s status={status} ratio={ratio:.2f} (continuacao)")
        if ratio > best_min_level:
            best_min_level = ratio
            best = (candidate, sr)
        if ratio >= quality_ratio:
            break
    if best is None:
        raise RuntimeError(f"Nao consegui gerar continuacao limpa para o prompt: {prompt}")
    trimmed, sr = best
    target_len = int(duration * sr)
    if len(trimmed) < target_len:
        trimmed = np.pad(trimmed, (0, target_len - len(trimmed)))
    return trimmed, sr


def normalize_rms(audio, target_rms=0.15, max_gain=4.0):
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < 1e-6:
        return audio
    gain = min(target_rms / rms, max_gain)
    return np.clip(audio * gain, -0.99, 0.99)


def make_loop_seamless(audio, sr, crossfade_s=2.0):
    fade_n = int(sr * crossfade_s)
    if len(audio) <= fade_n * 2:
        return audio, sr
    head = audio[:fade_n]
    tail = audio[-fade_n:]
    t = np.linspace(0, np.pi / 2, fade_n)
    blended = tail * np.cos(t) + head * np.sin(t)
    looped = np.concatenate([blended, audio[fade_n:-fade_n]])
    return looped, sr


def generate_loop_track(prompt, total_duration, segment_s=24, guidance_scale=4.5, seed=None, max_retries=3,
                         quality_ratio=0.3, continuation_s=3, loop_crossfade_s=2.0):
    print(f"  [1] bloco inicial ({min(segment_s, total_duration):.0f}s)...")
    audio, sr = generate_clean_segment(prompt, min(segment_s, total_duration), guidance_scale=guidance_scale,
                                        max_retries=max_retries, seed=seed, quality_ratio=quality_ratio)
    restante = total_duration - len(audio) / sr
    i = 2
    while restante > 0.5:
        bloco = min(segment_s, restante)
        print(f"  [{i}] continuacao (+{bloco:.0f}s)...")
        novo, sr = generate_continuation_segment(prompt, bloco, audio, sr, guidance_scale=guidance_scale,
                                                   max_retries=max_retries, seed=(seed + i * 13) if seed is not None else None,
                                                   quality_ratio=quality_ratio, continuation_s=continuation_s)
        audio = np.concatenate([audio, novo])
        restante = total_duration - len(audio) / sr
        i += 1
    audio = audio[:int(total_duration * sr)]
    looped, sr = make_loop_seamless(audio, sr, crossfade_s=loop_crossfade_s)
    return normalize_rms(looped, target_rms=0.15), sr


def to_stereo_int16(audio_mono):
    stereo = np.stack([audio_mono, audio_mono])
    return (stereo * 32767).astype(np.int16).T


print("Biblioteca carregada.")

## Conectar ao Google Drive (pra não perder o resultado quando a sessão do Colab acabar)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = '/content/drive/MyDrive/SAO_RPG_musicas_large'
os.makedirs(OUT_DIR, exist_ok=True)
print('Salvando em:', OUT_DIR)

## Gerar as 6 faixas do projeto (mesmos prompts estilo SAO/Kajiura)

Pode editar os prompts/durações abaixo se quiser, mas por padrão reproduz exatamente as 6 faixas do projeto local, só que com o `musicgen-large`.

In [ ]:
import scipy.io.wavfile

# 01_abertura -- tema estruturado (intro/verso/refrao), toca uma vez, nao precisa de loop
ABERTURA = [
    ("intro", "fast energetic instrumental intro hit, anime opening theme, bright synth brass stab, punchy rock drums, rising energy, distorted electric guitar accent, heroic and exciting, instrumental, no vocals", 16),
    ("verse", "upbeat anime opening verse, driving rock drums, groovy bass guitar, light electric guitar strumming, bright synth pads, energetic optimistic mood, instrumental, no vocals", 30),
    ("prechorus", "rising pre-chorus buildup, ascending synth arpeggio, snare drum roll buildup, increasing energy, anime opening theme, instrumental, no vocals", 16),
    ("chorus", "triumphant soaring anime opening chorus theme, powerful lead synth melody, big distorted electric guitar riff, driving rock drums, huge uplifting hook, heroic and joyful climax, instrumental, no vocals", 30),
    ("verse2", None, 30, "verse"),
    ("prechorus2", None, 16, "prechorus"),
    ("chorus2", None, 30, "chorus"),
    ("bridge", "instrumental bridge, energetic electric guitar solo, layered synths, driving rock drums, anime opening theme, exciting and heroic, instrumental, no vocals", 26),
    ("chorus3", None, 30, "chorus"),
    ("outro", "triumphant ending, big final hit, drum fill and cymbal crash, energetic finish, anime opening theme, instrumental, no vocals", 18),
]

# 02-06 -- loops de fundo, hibrido orquestral+eletronico estilo Kajiura, sem vocais
LOOP_TRACKS = {
    "02_ambiente": ("cinematic anime score, ethereal orchestral strings, gentle piano, subtle electronic texture underneath, bittersweet and wistful mood, moderate tempo, instrumental, no vocals", 100),
    "03_combate": ("cinematic anime battle score, driving orchestral string ostinato, electronic pulse layered underneath, tense and energetic, fast paced, instrumental, no vocals", 90),
    "04_combate_epico": ("epic cinematic anime battle score, huge orchestral string ostinato, aggressive electronic beat, dramatic and intense, relentless, instrumental, no vocals", 90),
    "05_combate_boss": ("dark cinematic anime boss battle score, dissonant orchestral strings ostinato, heavy electronic pulse, aggressive and relentless, epic scale, menacing, instrumental, no vocals", 90),
    "06_cidade": ("cinematic anime score, warm strings and piano, gentle, bittersweet undertone beneath a cozy melody, mid tempo, instrumental, no vocals", 100),
}

GUIDANCE = 4.5
RETRIES = 3


def build_abertura(seed_base=3000):
    print("\n===== 01_abertura =====")
    clips = {}
    ordered = []
    for i, (seg_id, prompt, dur, *reuse) in enumerate(ABERTURA, 1):
        if prompt is None:
            src = clips[reuse[0]]
            print(f"[{i}/{len(ABERTURA)}] {seg_id}: reaproveitado de {reuse[0]}")
            ordered.append(src)
            clips[seg_id] = src
            continue
        print(f"[{i}/{len(ABERTURA)}] Gerando {seg_id} ({dur}s)...")
        audio, sr = generate_clean_segment(prompt, dur, guidance_scale=GUIDANCE, max_retries=RETRIES, seed=seed_base + i * 17)
        clips[seg_id] = (audio, sr)
        ordered.append((audio, sr))
    sr = ordered[0][1]
    fade_n = int(sr * 0.8)
    result = normalize_rms(ordered[0][0]).copy()
    for audio, s in ordered[1:]:
        audio = normalize_rms(audio)
        t = np.linspace(0, np.pi / 2, fade_n)
        blended = result[-fade_n:] * np.cos(t) + audio[:fade_n] * np.sin(t)
        result = np.concatenate([result[:-fade_n], blended, audio[fade_n:]])
    return result, sr


def salvar(nome, audio, sr):
    path = os.path.join(OUT_DIR, f"{nome}.wav")
    scipy.io.wavfile.write(path, rate=sr, data=to_stereo_int16(audio))
    print(f"Salvo: {path} ({len(audio)/sr:.1f}s)")


t_start = time.time()

audio, sr = build_abertura()
salvar("01_abertura", audio, sr)

for idx, (nome, (prompt, duracao)) in enumerate(LOOP_TRACKS.items()):
    print(f"\n===== {nome} (loop, {duracao}s) =====")
    audio, sr = generate_loop_track(prompt, duracao, guidance_scale=GUIDANCE, seed=5000 + idx * 100, max_retries=RETRIES)
    salvar(nome, audio, sr)

print(f"\nTempo total: {(time.time()-t_start)/60:.1f} min")
print(f"Arquivos em: {OUT_DIR}")

## Depois de rodar

1. Abra `SAO_RPG_musicas_large/` no seu Google Drive.
2. Baixe os `.wav` (clique direito → Fazer download, ou baixe a pasta inteira zipada).
3. Substitua os arquivos correspondentes em `SAO RPG/musicas/` no seu PC (o projeto converte pra `.mp3` automaticamente só quando gerado pelo script local — aqui é só o `.wav`; se quiser o `.mp3`, converta com qualquer conversor ou me avise que eu rodo o ffmpeg localmente em cima do `.wav` baixado).